# Live Trading Data — Strategy Notebook

End-to-end walkthrough of consuming the live aggregation server from a Jupyter notebook.

What we'll do:
1. Connect to the local data server over WebSocket
2. Buffer **price ticks** into a `pandas.DataFrame` and chart them
3. Stream **news + GDELT events** and tag the relevant symbols
4. Watch **insider trades + bulk/block deals** for unusual activity
5. Combine all streams into a simple **multi-signal scanner**

Prerequisites:
```bash
# Terminal 1 — start the aggregation server
pip install -e .
data-server

# Notebook deps
pip install jupyter pandas matplotlib nest_asyncio
```

## 1.  Setup

We use `nest_asyncio` so that async coroutines can run inside notebook cells without spawning a new event loop.

In [ ]:
import asyncio
import sys
from collections import defaultdict, deque
from datetime import datetime
from pathlib import Path

import nest_asyncio
import orjson
import pandas as pd
import websockets
from IPython.display import clear_output, display
import matplotlib.pyplot as plt

# Make project importable from notebooks/ directory
sys.path.insert(0, str(Path.cwd().parent))

from server.models.base import MessageType, SubscribeMessage

nest_asyncio.apply()

SERVER_URL = "ws://localhost:8765"

## 2.  A reusable async data collector

`collect()` opens a WebSocket, subscribes to the requested channels/symbols, and runs a callback on every incoming message until a deadline (or message count) is reached.

In [ ]:
async def collect(callback, channels, symbols=None, duration_s=30, max_messages=None):
    """Subscribe and dispatch every DataMessage to `callback(msg_dict)`."""
    sub = SubscribeMessage(
        action="subscribe",
        channels=channels,
        symbols=symbols or [],
    )
    deadline = asyncio.get_event_loop().time() + duration_s
    count = 0

    async with websockets.connect(SERVER_URL) as ws:
        await ws.send(orjson.dumps(sub.model_dump(mode="json")))
        while True:
            timeout = deadline - asyncio.get_event_loop().time()
            if timeout <= 0:
                break
            try:
                raw = await asyncio.wait_for(ws.recv(), timeout=timeout)
            except asyncio.TimeoutError:
                break
            data = orjson.loads(raw)
            if data.get("type") in ("ack", "error", "info", "pong"):
                continue
            callback(data)
            count += 1
            if max_messages and count >= max_messages:
                break
    return count

## 3.  Buffer price ticks into a DataFrame

We listen for 60 s on a small watchlist and accumulate every tick into a list, then convert to a DataFrame for analysis.

In [ ]:
WATCHLIST = ["RELIANCE", "TCS", "INFY", "HDFCBANK", "ICICIBANK"]
ticks = []

def on_tick(msg):
    if msg["type"] == "price_tick":
        d = msg["data"]
        ticks.append({
            "timestamp": pd.to_datetime(d.get("trade_time") or msg["timestamp"]),
            "symbol": d["symbol"],
            "ltp": d["ltp"],
            "change_pct": d["change_pct"],
            "volume": d["volume"],
        })

n = await collect(on_tick, channels=[MessageType.PRICE_TICK], symbols=WATCHLIST, duration_s=60)
print(f"Received {n} ticks ({len(ticks)} price ticks) over 60s")

df = pd.DataFrame(ticks).sort_values("timestamp").reset_index(drop=True)
df.head()

### Per-symbol summary

In [ ]:
summary = (
    df.groupby("symbol")
      .agg(ticks=("ltp", "size"),
           open=("ltp", "first"),
           last=("ltp", "last"),
           high=("ltp", "max"),
           low=("ltp", "min"),
           total_volume=("volume", "max"))
      .assign(window_pct=lambda x: (x["last"] - x["open"]) / x["open"] * 100)
)
summary

### Plot intraday paths

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for sym, g in df.groupby("symbol"):
    if len(g) < 2:
        continue
    norm = g["ltp"] / g["ltp"].iloc[0] * 100
    ax.plot(g["timestamp"], norm, label=sym, linewidth=1.5)

ax.set_title("Normalised intraday paths (first tick = 100)")
ax.set_ylabel("Index value")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4.  Live console of news + GDELT events

Subscribe only to news/GDELT for the same watchlist and watch them stream in for 2 minutes.

In [ ]:
news_log = []

def on_news(msg):
    d = msg["data"]
    if msg["type"] == "news":
        news_log.append({
            "published_at": pd.to_datetime(d["published_at"]),
            "source": d["source"],
            "symbols": ",".join(d.get("symbols", [])) or "—",
            "title": d["title"][:100],
            "category": d.get("category", ""),
        })
    elif msg["type"] == "gdelt_event":
        news_log.append({
            "published_at": pd.to_datetime(d["seendate"]),
            "source": f"gdelt:{d['domain']}",
            "symbols": ",".join(d.get("symbols", [])) or "—",
            "title": d["title"][:100],
            "category": f"tone={d.get('tone'):+.1f}" if d.get("tone") is not None else "",
        })

n = await collect(
    on_news,
    channels=[MessageType.NEWS, MessageType.GDELT_EVENT],
    symbols=WATCHLIST,
    duration_s=120,
)
print(f"Captured {len(news_log)} news/event items")
pd.DataFrame(news_log).sort_values("published_at", ascending=False).head(20)

### Per-symbol news count

In [ ]:
news_df = pd.DataFrame(news_log)
(
    news_df["symbols"]
      .str.split(",")
      .explode()
      .pipe(lambda s: s[s != "—"])
      .value_counts()
      .head(10)
      .plot(kind="barh", figsize=(8, 4), title="News mentions in last 2 min")
)
plt.tight_layout()
plt.show()

## 5.  Insider trades + bulk/block deals

These messages arrive less frequently — bulk and block deal updates land every 15 minutes; SEBI insider filings arrive hourly. We collect them across all symbols (no symbol filter).

In [ ]:
insider_events = []

def on_insider(msg):
    d = msg["data"]
    insider_events.append({
        "timestamp": pd.to_datetime(msg["timestamp"]),
        "type": msg["type"],
        "symbol": d.get("symbol", ""),
        "name": d.get("acquirer_name") or d.get("client_name", ""),
        "side": d.get("transaction_type") or d.get("deal_type", ""),
        "quantity": d.get("shares") or d.get("quantity"),
        "value": d.get("value"),
        "price": d.get("price"),
    })

await collect(
    on_insider,
    channels=[
        MessageType.INSIDER_TRADE,
        MessageType.BULK_DEAL,
        MessageType.BLOCK_DEAL,
    ],
    duration_s=30,  # short window — server will emit cached events on connect
)

ins_df = pd.DataFrame(insider_events)
ins_df.sort_values("timestamp", ascending=False).head(15) if not ins_df.empty else "No insider events in window"

## 6.  Multi-signal scanner

A more realistic strategy pattern: maintain rolling state for **price**, **news**, and **insider** activity per symbol and emit a composite signal whenever multiple inputs agree.

Signal logic:
- Price: > 1 % from open on above-average volume → +1 momentum
- News: any item in the last 5 minutes → +1 attention
- Insider buy or bulk-deal BUY in last 30 min → +1 flow

Composite score ≥ 2 triggers a console alert.

In [ ]:
from datetime import timedelta

class Scanner:
    def __init__(self, symbols):
        self.symbols = set(symbols)
        self.prices = defaultdict(lambda: deque(maxlen=50))
        self.opens = {}
        self.last_news = {}
        self.last_insider_buy = {}
        self.alerts = []

    def on_message(self, msg):
        t = msg["type"]
        d = msg["data"]
        ts = pd.to_datetime(msg["timestamp"])

        if t == "price_tick":
            sym = d["symbol"]
            self.prices[sym].append((ts, d["ltp"], d["volume"]))
            if sym not in self.opens and d.get("open"):
                self.opens[sym] = d["open"]
            self._evaluate(sym, ts, d["ltp"], d["volume"])

        elif t == "news":
            for sym in d.get("symbols", []):
                if sym in self.symbols:
                    self.last_news[sym] = ts

        elif t in ("insider_trade", "bulk_deal", "block_deal"):
            sym = d.get("symbol", "")
            side = (d.get("transaction_type") or d.get("deal_type", "")).upper()
            if sym in self.symbols and "BUY" in side:
                self.last_insider_buy[sym] = ts

    def _evaluate(self, sym, ts, ltp, volume):
        if sym not in self.opens:
            return
        op = self.opens[sym]
        pct_from_open = (ltp - op) / op * 100

        # Volume signal: current vs rolling average
        recent_vols = [v for _, _, v in list(self.prices[sym])[-20:]]
        avg_vol = sum(recent_vols) / max(len(recent_vols), 1)

        score = 0
        reasons = []

        if abs(pct_from_open) > 1.0 and volume > avg_vol * 1.2:
            score += 1
            reasons.append(f"momentum {pct_from_open:+.2f}%")

        if sym in self.last_news and ts - self.last_news[sym] < timedelta(minutes=5):
            score += 1
            reasons.append("recent news")

        if sym in self.last_insider_buy and ts - self.last_insider_buy[sym] < timedelta(minutes=30):
            score += 1
            reasons.append("insider/bulk BUY")

        if score >= 2:
            self.alerts.append({
                "timestamp": ts,
                "symbol": sym,
                "ltp": ltp,
                "score": score,
                "reasons": ", ".join(reasons),
            })

scanner = Scanner(WATCHLIST)

await collect(
    scanner.on_message,
    channels=[
        MessageType.PRICE_TICK,
        MessageType.NEWS,
        MessageType.INSIDER_TRADE,
        MessageType.BULK_DEAL,
        MessageType.BLOCK_DEAL,
    ],
    symbols=WATCHLIST,
    duration_s=180,
)

alerts_df = pd.DataFrame(scanner.alerts)
print(f"{len(alerts_df)} composite signals fired")
alerts_df.tail(20)

## 7.  Where to go next

- Wire alerts into your broker's order API (e.g. GROWW REST `POST /v1/orders`).
- Replace the simple scoring with a learned model (logistic regression / gradient boosting on labelled outcomes).
- Add macro overlays from `MessageType.GDELT_EVENT` and `MessageType.INDEX_DATA`.
- Persist messages to Parquet / TimescaleDB for backtesting (the hub already broadcasts JSON — pipe a tap into your warehouse).

The strategy class in `client/base.py` is the production-grade equivalent of the `collect()` helper here — subclass it and override `on_*` handlers when you're ready to deploy a long-running strategy.